In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score, roc_auc_score, roc_curve
import gradio as gr
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from matplotlib.patches import Patch

# 抑制sklearn警告
warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('default')
sns.set_palette("husl")

# ==================== 配置 ====================
class Config:
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_PATH = './data/f1_model_final_v3.pth'
    DATA_DIR = './data/'
    
    CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
    NUM_COLS = ['year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
                'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
                'Driver_Prev_Season_FL_Count', 'Is_Home_Race', 'Recent_3_Races_Avg_Pos', 
                'Performance_Trend', 'Consistency_Score']
    TARGET_COL = 'is_winner'
    
    EMB_DIM = 32
    HIDDEN_DIM = 128
    DROPOUT_RATE = 0.4
    BATCH_SIZE = 256
    LEARNING_RATE = 5e-4
    N_EPOCHS = 50
    PATIENCE = 10

def set_seeds():
    """設定隨機種子"""
    np.random.seed(Config.SEED)
    torch.manual_seed(Config.SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(Config.SEED)

# ==================== 數據處理 ====================
def clean_string(text):
    """清理文字"""
    return re.sub(r'\s+', ' ', text).strip() if isinstance(text, str) else text

def get_country_from_gp(gp_name):
    """從GP名稱獲取國家代碼"""
    gp_map = {
        'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
        'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
        'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
        'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
        'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
        'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
    }
    if not isinstance(gp_name, str):
        return None
    for key, country in gp_map.items():
        if key in gp_name:
            return country
    return None

def load_data():
    """載入所有CSV數據"""
    d = Config.DATA_DIR
    
    # 讀取數據
    winners = pd.read_csv(d + 'winners.csv', encoding='utf-8')
    drivers = pd.read_csv(d + 'drivers_updated.csv', encoding='utf-8')
    teams = pd.read_csv(d + 'teams_updated.csv', encoding='utf-8')
    fastest_laps = pd.read_csv(d + 'fastest_laps_updated.csv', encoding='utf-8')
    
    # 清理文字欄位
    for df in [winners, drivers, teams, fastest_laps]:
        for col in df.select_dtypes(include='object'):
            df[col] = df[col].map(clean_string)
    
    # 處理年份
    winners['year'] = pd.to_datetime(winners['Date'], errors='coerce').dt.year.astype(int)
    drivers.rename(columns={'Car': 'Team'}, inplace=True)
    
    for df in [drivers, teams, fastest_laps]:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
        if 'Pos' in df.columns:
            df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
    
    # 排除Indianapolis 500
    winners = winners[~winners['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    fastest_laps = fastest_laps[~fastest_laps['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    
    return winners, drivers, teams, fastest_laps

def create_features(drivers, teams, fastest_laps):
    """創建所有特徵"""
    def_pos_drv, def_pos_team = 20, 10
    
    # 車手lag特徵
    drivers = drivers.sort_values(['Driver', 'year'])
    drivers['Prev_Year_Driver_PTS'] = drivers.groupby('Driver')['PTS'].shift(1).fillna(0)
    drivers['Prev_Year_Driver_Pos'] = drivers.groupby('Driver')['Pos'].shift(1).fillna(def_pos_drv)
    drivers['Driver_Experience_Years'] = drivers['year'] - drivers.groupby('Driver')['year'].transform('min')
    
    # 車隊lag特徵
    teams = teams.sort_values(['Team', 'year'])
    teams['Prev_Year_Team_PTS'] = teams.groupby('Team')['PTS'].shift(1).fillna(0)
    teams['Prev_Year_Team_Pos'] = teams.groupby('Team')['Pos'].shift(1).fillna(def_pos_team)
    teams['Team_Experience_Years'] = teams['year'] - teams.groupby('Team')['year'].transform('min')
    
    # 合併車隊特徵
    drivers = drivers.merge(teams[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']], 
                           on=['Team', 'year'], how='left').fillna(0)
    
    # 最快圈速特徵
    fl = fastest_laps.groupby(['year', 'Driver']).size().reset_index(name='FL_Count')
    fl['Driver_Prev_Season_FL_Count'] = fl.groupby('Driver')['FL_Count'].shift(1).fillna(0)
    drivers = drivers.merge(fl[['Driver', 'year', 'Driver_Prev_Season_FL_Count']], on=['Driver', 'year'], how='left').fillna(0)
    
    # Momentum特徵
    drivers['Recent_3_Races_Avg_Pos'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).mean().reset_index(0, drop=True).fillna(def_pos_drv)
    
    # 安全計算表現趨勢
    drivers['PTS_prev'] = drivers.groupby('Driver')['PTS'].shift(1)
    drivers['Performance_Trend'] = 0.0
    mask = (drivers['PTS_prev'].notna()) & (drivers['PTS_prev'] > 0)
    drivers.loc[mask, 'Performance_Trend'] = ((drivers.loc[mask, 'PTS'] - drivers.loc[mask, 'PTS_prev']) / drivers.loc[mask, 'PTS_prev']).clip(-2.0, 2.0)
    drivers.drop('PTS_prev', axis=1, inplace=True)
    
    # 一致性評分
    drivers['Pos_Rolling_Std'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).std().reset_index(0, drop=True).fillna(0.0)
    drivers['Consistency_Score'] = 1.0 / (1.0 + drivers['Pos_Rolling_Std'])
    
    # 確保數值安全
    for col in ['Recent_3_Races_Avg_Pos', 'Performance_Trend', 'Consistency_Score']:
        drivers[col] = drivers[col].replace([float('inf'), float('-inf')], 0.0).fillna(0.0)
    
    return drivers

def build_dataset(winners, drivers):
    """構建建模數據集"""
    data = []
    drivers_by_year = {y: g for y, g in drivers.groupby('year')}
    
    for _, race in winners.iterrows():
        year, gp, winner = race['year'], race['Grand Prix'], race['Winner']
        if year not in drivers_by_year:
            continue
            
        race_country = get_country_from_gp(gp)
        for _, driver in drivers_by_year[year].iterrows():
            is_home = 1 if race_country and driver['Nationality'] == race_country else 0
            
            data.append({
                'year': year, 'Grand Prix': gp, 'Driver': driver['Driver'], 
                'Team': driver['Team'], 'Nationality': driver['Nationality'],
                'Prev_Year_Driver_PTS': driver['Prev_Year_Driver_PTS'],
                'Prev_Year_Driver_Pos': driver['Prev_Year_Driver_Pos'],
                'Driver_Experience_Years': driver['Driver_Experience_Years'],
                'Prev_Year_Team_PTS': driver['Prev_Year_Team_PTS'],
                'Prev_Year_Team_Pos': driver['Prev_Year_Team_Pos'],
                'Team_Experience_Years': driver['Team_Experience_Years'],
                'Driver_Prev_Season_FL_Count': driver['Driver_Prev_Season_FL_Count'],
                'Recent_3_Races_Avg_Pos': driver['Recent_3_Races_Avg_Pos'],
                'Performance_Trend': driver['Performance_Trend'],
                'Consistency_Score': driver['Consistency_Score'],
                'Is_Home_Race': is_home,
                'is_winner': int(driver['Driver'] == winner)
            })
    
    return pd.DataFrame(data)

#數據預處理
def preprocess_data(train_df, test_df):
    
    # 填補缺失值
    for col in Config.CAT_COLS:
        train_df[col] = train_df[col].fillna('Unknown')
        test_df[col] = test_df[col].fillna('Unknown')
    for col in Config.NUM_COLS:
        train_df[col] = train_df[col].fillna(0)
        test_df[col] = test_df[col].fillna(0)
    
    # 編碼分類特徵
    encoders, cat_dims = {}, {}
    for col in Config.CAT_COLS:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        test_df[col] = test_df[col].map(lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else len(le.classes_))
        encoders[col] = le
        cat_dims[col] = len(le.classes_) + 1
    
    # 標準化數值特徵
    scaler = StandardScaler()
    train_df[Config.NUM_COLS] = scaler.fit_transform(train_df[Config.NUM_COLS].values)
    test_df[Config.NUM_COLS] = scaler.transform(test_df[Config.NUM_COLS].values)
    
    return train_df, test_df, encoders, scaler, cat_dims

# ==================== 模型 ====================
#Focal Loss處理類別不平衡
class FocalLoss(nn.Module):
    
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, pred, target):
        pred_sigmoid = torch.sigmoid(pred)
        target = target.view(-1, 1)
        ce_loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        p_t = pred_sigmoid * target + (1 - pred_sigmoid) * (1 - target)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * ce_loss
        return focal_loss.mean()

class F1Dataset(Dataset):
    def __init__(self, df):
        self.x_cat = df[Config.CAT_COLS].values
        self.x_num = df[Config.NUM_COLS].values
        self.y = df[Config.TARGET_COL].values
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (torch.tensor(self.x_cat[idx], dtype=torch.long),
                torch.tensor(self.x_num[idx], dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.float32))

class F1Model(nn.Module):
    def __init__(self, cat_dims, num_feats):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, Config.EMB_DIM) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_feats)
        self.fc = nn.Sequential(
            nn.Linear(len(cat_dims) * Config.EMB_DIM + num_feats, Config.HIDDEN_DIM),
            nn.ReLU(), nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM, Config.HIDDEN_DIM // 2),
            nn.ReLU(), nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM // 2, 1)
        )
    
    def forward(self, x_cat, x_num):
        cat_emb = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], 1)
        num_norm = self.bn_num(x_num)
        combined = torch.cat([cat_emb, num_norm], 1)
        return self.fc(combined)

#訓練模型
def train_model(model, train_loader, test_loader):
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    criterion = FocalLoss()
    
    best_loss = float('inf')
    no_improve = 0
    train_losses, test_losses = [], []
    
    for epoch in range(Config.N_EPOCHS):
        # 訓練
        model.train()
        train_loss = 0
        for x_cat, x_num, y in train_loader:
            x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_cat, x_num), y.unsqueeze(1))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # 驗證
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for x_cat, x_num, y in test_loader:
                x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
                test_loss += criterion(model(x_cat, x_num), y.unsqueeze(1)).item()
        
        train_loss /= len(train_loader)
        test_loss /= len(test_loader)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        
        print(f"Epoch {epoch+1} | Train: {train_loss:.4f} | Test: {test_loss:.4f}")
        
        # 早停
        if test_loss < best_loss:
            best_loss = test_loss
            torch.save(model.state_dict(), Config.MODEL_PATH)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= Config.PATIENCE:
                print("Early stopping")
                break
    
    # 保存訓練曲線
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train', linewidth=2)
    plt.plot(test_losses, label='Test', linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training and Validation Loss', fontsize=16, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('./data/loss_curve_final.png', dpi=300, bbox_inches='tight')
    plt.close()

# ==================== 評估和繪圖功能 ====================
#計算特徵
def calculate_feature_importance(model, test_loader, feature_names):
    
    model.eval()
    importances = []
    
    for x_cat, x_num, y in test_loader:
        x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
        x_num.requires_grad_(True)
        
        output = model(x_cat, x_num)
        loss = F.binary_cross_entropy_with_logits(output, y.unsqueeze(1))
        
        grad = torch.autograd.grad(loss, x_num, create_graph=False)[0]
        importances.append(grad.abs().mean(dim=0).cpu().detach().numpy())
    
    avg_importance = np.mean(importances, axis=0)
    return dict(zip(feature_names, avg_importance))

#計算每場比賽的Top-3準確率
def calculate_top3_accuracy_by_race(model, test_df):
    
    model.eval()
    race_groups = test_df.groupby(['year', 'Grand Prix'])
    top3_correct = 0
    total_races = 0
    
    for (year, gp), race_data in race_groups:
        # 獲取該場比賽的所有預測
        x_cat = torch.tensor(race_data[Config.CAT_COLS].values, dtype=torch.long).to(Config.DEVICE)
        x_num = torch.tensor(race_data[Config.NUM_COLS].values, dtype=torch.float32).to(Config.DEVICE)
        
        with torch.no_grad():
            probs = torch.sigmoid(model(x_cat, x_num)).cpu().numpy().flatten()
        
        # 找出真實獲勝者
        true_winner_idx = race_data[race_data[Config.TARGET_COL] == 1].index
        if len(true_winner_idx) == 0:
            continue
        
        # 找出預測概率最高的前3名
        race_indices = race_data.index.tolist()
        prob_with_idx = list(zip(probs, race_indices))
        prob_with_idx.sort(key=lambda x: x[0], reverse=True)
        top3_indices = [idx for _, idx in prob_with_idx[:3]]
        
        # 檢查真實獲勝者是否在前3名中
        if any(idx in top3_indices for idx in true_winner_idx):
            top3_correct += 1
        
        total_races += 1
    
    return top3_correct / total_races if total_races > 0 else 0

#繪製混淆矩陣
def plot_confusion_matrix(y_true, y_pred, save_path='./data/confusion_matrix.png'):
    
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Not Winner', 'Winner'],
                yticklabels=['Not Winner', 'Winner'])
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

#繪製ROC曲線
def plot_roc_curve(y_true, y_probs, save_path='./data/roc_curve.png'):
   
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'ROC Curve (AUC = {auc_score:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
             label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=16, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

#繪製特徵重要性圖
def plot_feature_importance(importance_dict, save_path='./data/feature_importance.png'):

    features = list(importance_dict.keys())
    importances = list(importance_dict.values())
    
    # 排序
    sorted_idx = np.argsort(importances)
    features_sorted = [features[i] for i in sorted_idx]
    importances_sorted = [importances[i] for i in sorted_idx]
    
    plt.figure(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0, 1, len(features_sorted)))
    bars = plt.barh(range(len(features_sorted)), importances_sorted, color=colors)
    plt.yticks(range(len(features_sorted)), features_sorted)
    plt.xlabel('Feature Importance (Gradient-based)', fontsize=12)
    plt.title('Feature Importance Analysis', fontsize=16, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    
    # 添加數值標籤
    for i, (bar, imp) in enumerate(zip(bars, importances_sorted)):
        plt.text(bar.get_width() + max(importances_sorted) * 0.01, 
                bar.get_y() + bar.get_height()/2, 
                f'{imp:.4f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

#繪製預測概率分布圖
def plot_prediction_distribution(y_true, y_probs, save_path='./data/prediction_distribution.png'):

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 左圖：按真實標籤分別顯示預測概率分布
    winners_probs = [prob for prob, label in zip(y_probs, y_true) if label == 1]
    non_winners_probs = [prob for prob, label in zip(y_probs, y_true) if label == 0]
    
    ax1.hist(non_winners_probs, bins=50, alpha=0.7, label='Non-Winners', 
             color='skyblue', density=True)
    ax1.hist(winners_probs, bins=50, alpha=0.7, label='Winners', 
             color='salmon', density=True)
    ax1.set_xlabel('Predicted Probability', fontsize=12)
    ax1.set_ylabel('Density', fontsize=12)
    ax1.set_title('Prediction Probability Distribution by True Label', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右圖：預測概率的整體分布
    ax2.hist(y_probs, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
    ax2.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')
    ax2.set_xlabel('Predicted Probability', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title('Overall Prediction Probability Distribution', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

#分析預測概率最高的樣本
def plot_top_predictions_analysis(y_true, y_probs, save_path='./data/top_predictions_analysis.png'):
    
    # 將概率和真實標籤配對並排序
    prob_label_pairs = list(zip(y_probs, y_true))
    prob_label_pairs.sort(key=lambda x: x[0], reverse=True)
    
    # 取前100個最高概率的預測
    top_n = min(100, len(prob_label_pairs))
    top_probs = [pair[0] for pair in prob_label_pairs[:top_n]]
    top_labels = [pair[1] for pair in prob_label_pairs[:top_n]]
    
    plt.figure(figsize=(12, 8))
    
    # 創建顏色映射
    colors = ['red' if label == 1 else 'blue' for label in top_labels]
    
    plt.scatter(range(top_n), top_probs, c=colors, alpha=0.6, s=50)
    plt.xlabel(f'Rank (Top {top_n} Predictions)', fontsize=12)
    plt.ylabel('Predicted Probability', fontsize=12)
    plt.title('Analysis of Top Predictions', fontsize=16, fontweight='bold')
    
    # 添加圖例
    legend_elements = [Patch(facecolor='red', label='True Winners'),
                      Patch(facecolor='blue', label='True Non-Winners')]
    plt.legend(handles=legend_elements)
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

#評估模型 - 加入Top-3準確率和Recall，並生成所有分析圖表
def evaluate_model(model, test_loader, test_df):

    if os.path.exists(Config.MODEL_PATH):
        model.load_state_dict(torch.load(Config.MODEL_PATH, map_location=Config.DEVICE))
    
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    
    with torch.no_grad():
        for x_cat, x_num, y in test_loader:
            x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
            probs = torch.sigmoid(model(x_cat, x_num))
            pred = (probs > 0.5).squeeze().int()
            
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy() if pred.ndim > 0 else [pred.item()])
            y_probs.extend(probs.cpu().numpy().flatten())
    
    # 基本準確率
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Binary Classification Accuracy: {accuracy:.4f}")
    
    # Recall計算
    recall = recall_score(y_true, y_pred, zero_division=0)
    print(f"Recall (Winner Detection): {recall:.4f}")
    
    # AUC計算
    auc_score = roc_auc_score(y_true, y_probs)
    print(f"AUC Score: {auc_score:.4f}")
    
    # Top-3準確率（按比賽分組計算）
    top3_accuracy = calculate_top3_accuracy_by_race(model, test_df)
    print(f"Top-3 Accuracy (Race-wise): {top3_accuracy:.4f}")
    
    # 詳細分類報告
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Not Winner', 'Winner'], zero_division=0))
    
    # 混淆矩陣
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    
    # ==================== 生成所有圖表 ====================
    print("\nGenerating analysis plots...")
    
    # 1. 混淆矩陣熱力圖
    plot_confusion_matrix(y_true, y_pred)
    print("✓ Confusion matrix heatmap saved")
    
    # 2. ROC曲線
    plot_roc_curve(y_true, y_probs)
    print("✓ ROC curve saved")
    
    # 3. 特徵重要性
    try:
        importance_dict = calculate_feature_importance(model, test_loader, Config.NUM_COLS)
        plot_feature_importance(importance_dict)
        print("✓ Feature importance plot saved")
    except Exception as e:
        print(f"⚠ Feature importance calculation failed: {e}")
    
    # 4. 預測概率分布
    plot_prediction_distribution(y_true, y_probs)
    print("✓ Prediction distribution plots saved")
    
    # 5. 頂部預測分析
    plot_top_predictions_analysis(y_true, y_probs)
    print("✓ Top predictions analysis saved")
    
    # 輸出各項數值
    print("\n" + "="*50)
    print("EVALUATION SUMMARY")
    print("="*50)
    print(f"Binary Classification Accuracy: {accuracy:.4f}")
    print(f"Top-3 Accuracy (Race-wise): {top3_accuracy:.4f}")
    print(f"Recall (Winner Detection): {recall:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    print("="*50)
    print("\nAll analysis plots have been saved to ./data/ directory")

# ==================== 預測界面 ====================
class F1Predictor:
    def __init__(self):
        self.model = None
        self.encoders = {}
        self.scaler = None
        self.driver_data = {}
        self.years = []
        self.gps = []
    
    def setup(self, model, encoders, scaler, drivers, winners):
        
        self.model = model
        self.encoders = encoders
        self.scaler = scaler
        
        for year, group in drivers.groupby('year'):
            self.driver_data[year] = group.to_dict('records')
        
        self.years = sorted(drivers['year'].unique())
        self.gps = sorted(winners['Grand Prix'].unique())
    
    #預測獲勝概率
    def predict(self, year_input, gp_input):
    
        try:
            year = int(year_input)
        except:
            return "Error: Invalid year"
        
        if not gp_input or year not in self.driver_data:
            return "Error: No data available"
        
        self.model.eval()
        results = []
        
        for driver_info in self.driver_data[year]:
            # 分類特徵
            cat = []
            for col in Config.CAT_COLS:
                val = str(driver_info.get(col, 'Unknown'))
                if col == 'Grand Prix':
                    val = gp_input
                
                if val in self.encoders[col].classes_:
                    encoded = self.encoders[col].transform([val])[0]
                else:
                    encoded = len(self.encoders[col].classes_)
                cat.append(encoded)
            
            # 數值特徵
            num = []
            for col in Config.NUM_COLS:
                if col == 'Is_Home_Race':
                    race_country = get_country_from_gp(gp_input)
                    driver_nationality = driver_info.get('Nationality', '')
                    val = 1.0 if race_country and driver_nationality == race_country else 0.0
                else:
                    val = float(driver_info.get(col, 0))
                num.append(val)
            
            # 預測
            x_num = torch.tensor(self.scaler.transform(np.array([num])), dtype=torch.float32).to(Config.DEVICE)
            x_cat = torch.tensor([cat], dtype=torch.long).to(Config.DEVICE)
            
            with torch.no_grad():
                prob = torch.sigmoid(self.model(x_cat, x_num)).cpu().item()
            
            results.append((driver_info.get('Driver', 'N/A'), driver_info.get('Team', 'N/A'), prob))
        
        # 排序並回傳前5名
        results.sort(key=lambda x: x[2], reverse=True)
        output = f"Predictions for {gp_input}, {year}:\n"
        for i, (driver, team, prob) in enumerate(results[:5], 1):
            output += f"{i}. {driver} ({team}): {prob:.2%}\n"
        
        return output.strip()
    
    #創建介面
    def create_interface(self):
        
        with gr.Blocks(theme=gr.themes.Soft()) as demo:
            gr.Markdown("# F1 Grand Prix Winner Predictor")
            
            with gr.Row():
                year_dd = gr.Dropdown(label="Year", choices=self.years, value=self.years[-1] if self.years else None)
                gp_dd = gr.Dropdown(label="Grand Prix", choices=self.gps, value=self.gps[0] if self.gps else None)
            
            predict_btn = gr.Button("Predict Winners")
            output_tb = gr.Textbox(label="Top 5 Predictions", lines=6, interactive=False)
            
            predict_btn.click(self.predict, inputs=[year_dd, gp_dd], outputs=[output_tb])
            
            with gr.Accordion("Training Analysis & Results", open=False):
                with gr.Row():
                    with gr.Column():
                        if os.path.exists("./data/loss_curve_final.png"):
                            gr.Image(value="./data/loss_curve_final.png", label="Training Loss Curve")
                        if os.path.exists("./data/confusion_matrix.png"):
                            gr.Image(value="./data/confusion_matrix.png", label="Confusion Matrix")
                    
                    with gr.Column():
                        if os.path.exists("./data/roc_curve.png"):
                            gr.Image(value="./data/roc_curve.png", label="ROC Curve")
                        if os.path.exists("./data/feature_importance.png"):
                            gr.Image(value="./data/feature_importance.png", label="Feature Importance")
                
                with gr.Row():
                    if os.path.exists("./data/prediction_distribution.png"):
                        gr.Image(value="./data/prediction_distribution.png", label="Prediction Distribution Analysis")
                
                with gr.Row():
                    if os.path.exists("./data/top_predictions_analysis.png"):
                        gr.Image(value="./data/top_predictions_analysis.png", label="Top Predictions Analysis")
        
        return demo

# ==================== 主程序 ====================
def main():
    # 環境設定
    os.makedirs(Config.DATA_DIR, exist_ok=True)
    set_seeds()
    
    # 數據載入和特徵工程
    winners, drivers, teams, fastest_laps = load_data()
    drivers_feat = create_features(drivers, teams, fastest_laps)
    modeling_df = build_dataset(winners, drivers_feat)
    
    # 數據分割
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()
    
    if len(unique_race_ids) < 2:
        train_df = test_df = modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=Config.SEED)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    
    # 移除race_id並預處理
    for df in [train_df, test_df]:
        if 'race_id' in df.columns:
            df.drop(columns=['race_id'], inplace=True)
    
    train_df, test_df, encoders, scaler, cat_dims = preprocess_data(train_df, test_df)
    
    # 創建數據加載
    train_set = F1Dataset(train_df)
    test_set = F1Dataset(test_df)
    
    weights = [1. / (train_df[Config.TARGET_COL].value_counts().get(t, 1) + 1e-6) for t in train_df[Config.TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    
    train_loader = DataLoader(train_set, batch_size=Config.BATCH_SIZE, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # 模型訓練
    cat_dims_ordered = [cat_dims[c] for c in Config.CAT_COLS]
    model = F1Model(cat_dims_ordered, len(Config.NUM_COLS)).to(Config.DEVICE)
    
    if not os.path.exists(Config.MODEL_PATH):
        train_model(model, train_loader, test_loader)
    
    # 評估模型
    evaluate_model(model, test_loader, test_df)
    
    # 啟動界面
    predictor = F1Predictor()
    predictor.setup(model, encoders, scaler, drivers_feat, winners)
    demo = predictor.create_interface()
    demo.launch(share=False)

if __name__ == '__main__':
    main()

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1 | Train: 0.1343 | Test: 0.1007
Epoch 2 | Train: 0.0944 | Test: 0.0883
Epoch 3 | Train: 0.0827 | Test: 0.0866
Epoch 4 | Train: 0.0778 | Test: 0.0804
Epoch 5 | Train: 0.0737 | Test: 0.0857
Epoch 6 | Train: 0.0687 | Test: 0.0812
Epoch 7 | Train: 0.0664 | Test: 0.0800
Epoch 8 | Train: 0.0630 | Test: 0.0803
Epoch 9 | Train: 0.0613 | Test: 0.0806
Epoch 10 | Train: 0.0595 | Test: 0.0779
Epoch 11 | Train: 0.0549 | Test: 0.0818
Epoch 12 | Train: 0.0547 | Test: 0.0761
Epoch 13 | Train: 0.0528 | Test: 0.0775
Epoch 14 | Train: 0.0505 | Test: 0.0857
Epoch 15 | Train: 0.0488 | Test: 0.0845
Epoch 16 | Train: 0.0468 | Test: 0.0853
Epoch 17 | Train: 0.0462 | Test: 0.0856
Epoch 18 | Train: 0.0441 | Test: 0.0853
Epoch 19 | Train: 0.0445 | Test: 0.0858
Epoch 20 | Train: 0.0427 | Test: 0.0905
Epoch 21 | Train: 0.0401 | Test: 0.0942
Epoch 22 | Train: 0.0423 | Test: 0.0930
Early stopping
Binary Classification Accuracy: 0.8797
Recall (Winner Detection): 0.7500
AUC Score: 0.9110
Top-3 Accuracy (Race-wi